# Representations

How each cluster becomes a **single profile**, set with `ClusterConfig(representation=...)`.

| representation | the profile is… | preserves |
|----------------|-----------------|-----------|
| `medoid` | a real period from the cluster | realistic shape |
| `mean` | the average of the cluster | central tendency |
| `maxoid` | the most extreme period | peaks |
| `distribution` | reshaped to match the value histogram | the **duration curve** |
| `distribution_minmax` | distribution + each attribute's cluster min/max | distribution **and** extremes |
| `minmax_mean` | mean, but min/max kept per timestep | extremes around the average |

**There is no single default** — it follows the clustering method:

| method | default representation |
|---|---|
| `hierarchical` (default), `kmedoids`, `contiguous` | `medoid` |
| `kmeans`, `averaging` | `mean` |
| `kmaxoids` | `maxoid` |

So switching to `kmeans` silently switches you to `mean` as well. Setting `representation=`
explicitly overrides that for any method — the two levers are independent.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## They score differently on different things

**RMSE** measures point-by-point timing; **RMSE on the duration curve** measures how well the
*spread* of values is kept. A representation can win one and lose the other:

In [ ]:
from tsam import ClusterConfig

reps = [
    "medoid",
    "mean",
    "maxoid",
    "distribution",
    "distribution_minmax",
    "minmax_mean",
]
rows = {}
for rep in reps:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    rows[rep] = {
        "RMSE": round(float(r.accuracy.rmse.mean()), 4),
        "RMSE (duration curve)": round(float(r.accuracy.rmse_duration.mean()), 4),
    }
pd.DataFrame(rows).T

## On the duration curve

Sorting every value high-to-low ignores *when* things happen and shows the distribution. `mean`
and `medoid` shave the peaks and fill the valleys; `distribution` is built to trace this curve:

In [ ]:
import plotly.express as px

frames = []
for rep in ["medoid", "mean", "distribution", "minmax_mean"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    s = r.reconstructed["Load"].sort_values(ascending=False).reset_index(drop=True)
    frames.append(
        pd.DataFrame({"rank": range(len(s)), "Load": s.values, "representation": rep})
    )
orig = data["Load"].sort_values(ascending=False).reset_index(drop=True)
frames.append(
    pd.DataFrame(
        {"rank": range(len(orig)), "Load": orig.values, "representation": "original"}
    )
)
px.line(
    pd.concat(frames),
    x="rank",
    y="Load",
    color="representation",
    title="Duration curve by representation",
)

## What `mean` clips

Averaging can lower the maximum. `distribution` also averages duration-curve levels;
use `distribution_minmax` (or `Distribution(preserve_minmax=True)`) to retain the cluster's
minimum and maximum where feasible:

In [ ]:
bounds = {"original": [data["Load"].max(), data["Load"].min()]}
for rep in ["mean", "distribution", "distribution_minmax"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    bounds[rep] = [r.reconstructed["Load"].max(), r.reconstructed["Load"].min()]
pd.DataFrame(bounds, index=["peak Load", "min Load"]).round(1)

## Choose how attributes coincide

A duration curve describes each attribute separately. It does not tell you whether high load
occurs at the same time as low solar generation. For that, configure the **ordering** of the
distribution values with `Distribution(concurrency=...)`.

Start by comparing the default `independent` ordering with `medoid`, which borrows each
attribute's ranks from the same real member period. Keep the clustering and other settings
fixed. Here we use solar (`GHI`) and `Load` from the data above and turn off rescaling to
inspect the representations directly:

In [ ]:
from tsam import Distribution

joint_results = {}
for strategy in ["independent", "medoid"]:
    joint_results[strategy] = tsam.aggregate(
        data[["GHI", "Load"]],
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=Distribution(concurrency=strategy)),
        preserve_column_means=False,
    )

pd.DataFrame(
    {
        strategy: {
            "RMSE (duration curve)": result.accuracy.weighted_rmse_duration,
            "Pearson correlation error": result.concurrency.correlation_error,
            "Spearman correlation error": result.concurrency.rank_correlation_error,
        }
        for strategy, result in joint_results.items()
    }
).T.round(4)

For these fixed settings, the duration-curve errors are equal: ordering permutes the same
values within each cluster. This does **not** mean the compressed duration curves match the
original exactly. The two correlation errors compare the original and reconstructed series'
correlation matrices; lower is better. Here `medoid` reduces both correlation errors. Check
both on your data before choosing an ordering.

`Distribution(concurrency="medoid")` still constructs new values. It is different from
`representation="medoid"`, which selects a member period. Neither an exact original day nor
exact correlations are guaranteed by the distribution option.

### Available orderings

Pass one of these objects as `ClusterConfig(representation=...)`:

| Configuration | How values are ordered | Use when… |
|---|---|---|
| `Distribution()` | Each attribute follows its own mean profile (`independent`, the default). | Each attribute's average timing is the priority. |
| `Distribution(concurrency="medoid")` | Each attribute follows its ranks in the same cluster medoid. | You want to try a real period's pattern of co-occurrence. |
| `Distribution(concurrency="reference", reference_attribute="GHI")` | Every attribute follows the reference column's mean-profile ranks. | All attributes should rise and fall with that reference. |
| `Distribution(concurrency="consensus")` | All attributes share ranks derived from the first principal component of their standardized mean profiles. | You want a shared ordering without choosing a reference. |
| `Distribution(concurrency="assignment")` | All attributes share the ordering that minimizes total squared deviation from the cluster mean profile. | You want the closest fit to the mean under a shared-order constraint. |

The last three options place low values together and high values together within each typical
period. That can introduce positive correlations, especially when attributes originally move
in opposite directions. `medoid` allows different rank patterns per attribute, but it is not
guaranteed to give the lowest error on every dataset.

### Combine options and respect their scope

To also retain cluster extremes, use
`Distribution(concurrency="medoid", preserve_minmax=True)`. This fits the duration curve
while retaining the mean where feasible; very short periods or infeasible bounds can prevent
simultaneous preservation of the mean and extremes.

The non-default concurrency strategies require `scope="local"` (the default). A `reference`
strategy also requires `reference_attribute` to name an input column; supplying that column
without `concurrency` selects `reference` automatically.

Set concurrency on **`ClusterConfig`**. It is not supported on `SegmentConfig`, whose
representative is only one value per attribute. Subsequent segmentation can change the
duration curves and correlations, so measure the final result if you use both steps.

For a full-year comparison of fitting scope, extrema, and all five orderings, follow
[Duration representations](../tutorials/duration_representations.ipynb).

The basic representation rules also apply to [segments](segmentation.ipynb) through
`SegmentConfig(representation=...)`, subject to the distribution restrictions above.

---

* [Extreme periods](extreme_periods.ipynb) — force one *specific* period to survive verbatim.
* [Clustering methods](clustering_methods.ipynb) — the companion lever.
* [Comparing representations](../tutorials/comparing_representations.ipynb) — whole-day selection and
  coordinate-wise construction on one small cluster.